In [1]:
from pathlib import Path
import time

import numpy as np
import tensorflow as tf

from google.colab import drive


# MOUNT GOOGLE DRIVE
drive.mount("/content/drive")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Baseline MLP for Environmental Noise Cancellation
# Aligned with T-MLP and GRU
import os
import time
import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint


# 1. CONFIGURATION
# Google Drive dataset path
DATASET_PATH = "/content/drive/MyDrive/T-MLP/processed_data"

TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "validation")
TEST_PATH = os.path.join(DATASET_PATH, "test")

# Saved model directory
MODEL_DIR = "/content/drive/MyDrive/T-MLP/saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "MLP_best.keras"
)

# Audio / window configuration
SAMPLE_RATE = 16000
WINDOW_SIZE = 256
HOP_SIZE = 128

# Training configuration
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
CLIPNORM = 1.0

# Reproducibility
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# 2. ENVIRONMENT
print("=" * 60)
print("MLP ENVIRONMENT")
print("=" * 60)

print("Experiment: MLP_baseline")
print(f"TensorFlow: {tf.__version__}")
print(f"Random seed: {SEED}")

print("\nAvailable GPUs:")
print(tf.config.list_physical_devices("GPU"))

print("\nDeterministic operations: ENABLED")

# 3. DATASET VERIFICATION
def get_npz_files(folder):

    if not os.path.exists(folder):
        raise FileNotFoundError(
            f"Folder not found:\n{folder}"
        )

    files = [
        os.path.join(folder, file)
        for file in os.listdir(folder)
        if file.endswith(".npz")
    ]

    return sorted(files)


train_files = get_npz_files(TRAIN_PATH)
val_files = get_npz_files(VAL_PATH)
test_files = get_npz_files(TEST_PATH)


print("\n" + "=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

print(f"Training samples   : {len(train_files)}")
print(f"Validation samples : {len(val_files)}")
print(f"Test samples       : {len(test_files)}")


if len(train_files) != 800:
    raise ValueError(
        f"Expected 800 training samples, found {len(train_files)}"
    )

if len(val_files) != 100:
    raise ValueError(
        f"Expected 100 validation samples, found {len(val_files)}"
    )

if len(test_files) != 100:
    raise ValueError(
        f"Expected 100 test samples, found {len(test_files)}"
    )

print("\nDataset verification successful.")

# 4. LOAD ONE SAMPLE
def load_npz_sample(file_path):

    data = np.load(file_path)

    noisy = data["noisy_speech"].astype(np.float32)
    clean = data["clean_speech"].astype(np.float32)

    return noisy, clean

# 5. CREATE SLIDING WINDOWS
def create_windows(noisy, clean):

    noisy_windows = []
    clean_windows = []

    if len(noisy) != len(clean):
        raise ValueError(
            "Noisy and clean audio lengths do not match."
        )

    if len(noisy) < WINDOW_SIZE:

        return (
            np.empty(
                (0, WINDOW_SIZE),
                dtype=np.float32
            ),
            np.empty(
                (0, WINDOW_SIZE),
                dtype=np.float32
            )
        )

    for start in range(
        0,
        len(noisy) - WINDOW_SIZE + 1,
        HOP_SIZE
    ):

        end = start + WINDOW_SIZE

        noisy_windows.append(
            noisy[start:end]
        )

        clean_windows.append(
            clean[start:end]
        )

    return (
        np.asarray(
            noisy_windows,
            dtype=np.float32
        ),
        np.asarray(
            clean_windows,
            dtype=np.float32
        )
    )

# 6. LOAD COMPLETE DATASET
def load_dataset(file_list, dataset_name):

    print(
        f"\nLoading {dataset_name} dataset..."
    )

    all_noisy = []
    all_clean = []

    for index, file_path in enumerate(file_list):

        noisy, clean = load_npz_sample(
            file_path
        )

        noisy_windows, clean_windows = create_windows(
            noisy,
            clean
        )

        if len(noisy_windows) == 0:

            print(
                f"Warning: skipped "
                f"{os.path.basename(file_path)} "
                f"(too short)"
            )

            continue

        all_noisy.append(noisy_windows)
        all_clean.append(clean_windows)

        if (index + 1) % 100 == 0:

            print(
                f"Loaded {index + 1}/{len(file_list)} files"
            )

    if not all_noisy:

        raise ValueError(
            f"No valid windows found in "
            f"{dataset_name} dataset."
        )

    X = np.concatenate(
        all_noisy,
        axis=0
    )

    y = np.concatenate(
        all_clean,
        axis=0
    )

    print(
        f"{dataset_name} windows: {len(X)}"
    )

    return X, y

# 7. LOAD TRAIN / VALIDATION / TEST
X_train, y_train = load_dataset(
    train_files,
    "training"
)

X_val, y_val = load_dataset(
    val_files,
    "validation"
)

X_test, y_test = load_dataset(
    test_files,
    "test"
)

# 8. DATA NUMERICAL VALIDATION
print("\n" + "=" * 60)
print("DATA NUMERICAL VALIDATION")
print("=" * 60)

print(
    f"X_train: shape={X_train.shape}, "
    f"dtype={X_train.dtype}"
)

print(
    f"y_train: shape={y_train.shape}, "
    f"dtype={y_train.dtype}"
)

print(
    f"X_val: shape={X_val.shape}, "
    f"dtype={X_val.dtype}"
)

print(
    f"y_val: shape={y_val.shape}, "
    f"dtype={y_val.dtype}"
)

print(
    f"X_test: shape={X_test.shape}, "
    f"dtype={X_test.dtype}"
)

print(
    f"y_test: shape={y_test.shape}, "
    f"dtype={y_test.dtype}"
)

if not np.isfinite(X_train).all():
    raise ValueError("X_train contains NaN/Inf")

if not np.isfinite(y_train).all():
    raise ValueError("y_train contains NaN/Inf")

if not np.isfinite(X_val).all():
    raise ValueError("X_val contains NaN/Inf")

if not np.isfinite(y_val).all():
    raise ValueError("y_val contains NaN/Inf")

if not np.isfinite(X_test).all():
    raise ValueError("X_test contains NaN/Inf")

if not np.isfinite(y_test).all():
    raise ValueError("y_test contains NaN/Inf")

print("\nNaN/Inf validation: PASSED")

# 9. WINDOWED DATASET SUMMARY
print("\n" + "=" * 60)
print("WINDOWED DATASET")
print("=" * 60)

print(
    f"Training windows   : {X_train.shape}"
)

print(
    f"Validation windows : {X_val.shape}"
)

print(
    f"Test windows       : {X_test.shape}"
)

print(
    f"\nInput window size  : {WINDOW_SIZE} samples"
)

print(
    f"Sample rate        : {SAMPLE_RATE} Hz"
)

print(
    f"Window duration    : "
    f"{WINDOW_SIZE / SAMPLE_RATE * 1000:.2f} ms"
)

print(
    f"Hop size           : {HOP_SIZE} samples"
)

# 10. BUILD BASELINE MLP
def build_mlp():

    model = Sequential([

        Input(
            shape=(WINDOW_SIZE,),
            name="noisy_audio"
        ),

        Dense(
            512,
            activation="relu",
            name="dense_512"
        ),

        Dense(
            256,
            activation="relu",
            name="dense_256"
        ),

        Dense(
            WINDOW_SIZE,
            activation="linear",
            name="output"
        )
    ])

    optimizer = Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=CLIPNORM
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model

# 11. CREATE MODEL
print("\n" + "=" * 60)
print("BUILDING MLP")
print("=" * 60)

model = build_mlp()

model.summary()

print(
    f"\nTotal parameters: "
    f"{model.count_params():,}"
)

# 12. CONFIGURE CHECKPOINT
checkpoint = ModelCheckpoint(

    MODEL_PATH,

    monitor="val_loss",

    mode="min",

    save_best_only=True,

    save_weights_only=False,

    verbose=1
)


print("\n" + "=" * 60)
print("CONFIGURING CHECKPOINT")
print("=" * 60)

print("Monitoring: val_loss")
print("Mode: minimize")
print("Save best only: YES")

print(
    f"Best model path: {MODEL_PATH}"
)

# 13. TRAIN MODEL
print("\n" + "=" * 60)
print("STARTING MLP TRAINING")
print("=" * 60)

print(f"Epochs             : {EPOCHS}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Learning rate      : {LEARNING_RATE}")
print(f"Gradient clipnorm  : {CLIPNORM}")

training_start = time.perf_counter()

history = model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    shuffle=True,

    callbacks=[
        checkpoint
    ],

    verbose=1
)

training_end = time.perf_counter()

training_time = (
    training_end - training_start
)

# 14. TRAINING COMPLETED
print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)

print(
    f"Epochs trained: {len(history.history['loss'])}"
)

# 15. LOAD BEST MODEL
print("\n" + "=" * 60)
print("LOADING BEST MLP MODEL")
print("=" * 60)

if not os.path.exists(MODEL_PATH):

    raise FileNotFoundError(
        "Best MLP model was not created."
    )

best_model = tf.keras.models.load_model(
    MODEL_PATH
)

print(
    "Best MLP model loaded successfully."
)

print(
    f"Path: {MODEL_PATH}"
)

print(
    f"Parameters: {best_model.count_params():,}"
)

# 16. TEST EVALUATION
print("\n" + "=" * 60)
print("TEST EVALUATION")
print("=" * 60)

test_loss, test_mae = best_model.evaluate(

    X_test,
    y_test,

    batch_size=BATCH_SIZE,

    verbose=1
)

test_mse = test_loss

test_rmse = np.sqrt(
    test_mse
)


print(
    f"\nTest MSE  : {test_mse:.8f}"
)

print(
    f"Test RMSE : {test_rmse:.8f}"
)

print(
    f"Test MAE  : {test_mae:.8f}"
)

# 17. GENERATE TEST PREDICTIONS
print("\n" + "=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)

prediction_start = time.perf_counter()

predictions = best_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

prediction_end = time.perf_counter()

prediction_time = (
    prediction_end - prediction_start
)

print(
    f"Prediction time: "
    f"{prediction_time:.4f} seconds"
)

print(
    f"Prediction shape: "
    f"{predictions.shape}"
)

# 18. PREDICTION VALIDATION
print("\n" + "=" * 60)
print("PREDICTION VALIDATION")
print("=" * 60)

if not np.isfinite(predictions).all():

    raise ValueError(
        "Predictions contain NaN/Inf."
    )

print("NaN/Inf check: PASSED")

print(
    f"Prediction min : "
    f"{predictions.min():.6f}"
)

print(
    f"Prediction max : "
    f"{predictions.max():.6f}"
)

print(
    f"Prediction mean: "
    f"{predictions.mean():.6f}"
)

print(
    f"Prediction std : "
    f"{predictions.std():.6f}"
)

# 19. INFERENCE LATENCY
print("\n" + "=" * 60)
print("MEASURING INFERENCE LATENCY")
print("=" * 60)

# Warm-up
_ = best_model.predict(
    X_test[:1],
    verbose=0
)

latencies = []

num_latency_samples = min(
    100,
    len(X_test)
)

for i in range(num_latency_samples):

    sample = X_test[
        i:i + 1
    ]

    start = time.perf_counter()

    _ = best_model.predict(
        sample,
        verbose=0
    )

    end = time.perf_counter()

    latency_ms = (
        end - start
    ) * 1000

    latencies.append(
        latency_ms
    )


latencies = np.asarray(
    latencies
)

mean_latency = np.mean(
    latencies
)

median_latency = np.median(
    latencies
)

min_latency = np.min(
    latencies
)

max_latency = np.max(
    latencies
)


print(
    f"Samples measured : "
    f"{num_latency_samples}"
)

print(
    f"Mean latency     : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median latency   : "
    f"{median_latency:.4f} ms"
)

print(
    f"Minimum latency  : "
    f"{min_latency:.4f} ms"
)

print(
    f"Maximum latency  : "
    f"{max_latency:.4f} ms"
)

# 20. MODEL INFORMATION
print("\n" + "=" * 60)
print("MODEL INFORMATION")
print("=" * 60)

parameter_count = best_model.count_params()

model_size_bytes = os.path.getsize(
    MODEL_PATH
)

model_size_kb = (
    model_size_bytes / 1024
)

model_size_mb = (
    model_size_bytes / (1024 ** 2)
)


print(
    f"Parameters : {parameter_count:,}"
)

print(
    f"Model size : {model_size_kb:.2f} KB"
)

print(
    f"Model size : {model_size_mb:.4f} MB"
)

# 21. FINAL RESULTS
print("\n" + "=" * 60)
print("MLP FINAL RESULTS")
print("=" * 60)

print(
    f"Parameters       : "
    f"{parameter_count:,}"
)

print(
    f"MSE              : "
    f"{test_mse:.8f}"
)

print(
    f"RMSE             : "
    f"{test_rmse:.8f}"
)

print(
    f"MAE              : "
    f"{test_mae:.8f}"
)

print(
    f"Mean Latency     : "
    f"{mean_latency:.4f} ms"
)

print(
    f"Median Latency   : "
    f"{median_latency:.4f} ms"
)

print(
    f"Model Size       : "
    f"{model_size_kb:.2f} KB"
)

print(
    f"Training Time    : "
    f"{training_time:.2f} seconds"
)

print(
    f"Best Model       : "
    f"{MODEL_PATH}"
)

print("=" * 60)
print("MLP TRAINING COMPLETE")
print("=" * 60)

MLP ENVIRONMENT
Experiment: MLP_baseline
TensorFlow: 2.21.0
Random seed: 42

Available GPUs:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Deterministic operations: ENABLED

DATASET VERIFICATION
Training samples   : 800
Validation samples : 100
Test samples       : 100

Dataset verification successful.

Loading training dataset...
Loaded 100/800 files
Loaded 200/800 files
Loaded 300/800 files
Loaded 400/800 files
Loaded 500/800 files
Loaded 600/800 files
Loaded 700/800 files
Loaded 800/800 files
training windows: 732388

Loading validation dataset...
Loaded 100/100 files
validation windows: 80583

Loading test dataset...
Loaded 100/100 files
test windows: 85649

DATA NUMERICAL VALIDATION
X_train: shape=(732388, 256), dtype=float32
y_train: shape=(732388, 256), dtype=float32
X_val: shape=(80583, 256), dtype=float32
y_val: shape=(80583, 256), dtype=float32
X_test: shape=(85649, 256), dtype=float32
y_test: shape=(85649, 256), dtype=float32

NaN/Inf validation: PASSED

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_512 (Dense)               │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_256 (Dense)               │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 256)            │        65,792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 328,704 (1.25 MB)

 Trainable params: 328,704 (1.25 MB)

 Non-trainable params: 0 (0.00 B)


Total parameters: 328,704

CONFIGURING CHECKPOINT
Monitoring: val_loss
Mode: minimize
Save best only: YES
Best model path: /content/drive/MyDrive/T-MLP/saved_models/MLP_best.keras

STARTING MLP TRAINING
Epochs             : 10
Batch size         : 32
Learning rate      : 0.001
Gradient clipnorm  : 1.0
Epoch 1/10
22888/22888 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0019 - mae: 0.0248
Epoch 1: val_loss improved from None to 0.00142, saving model to /content/drive/MyDrive/T-MLP/saved_models/MLP_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/T-MLP/saved_models/MLP_best.keras
22888/22888 ━━━━━━━━━━━━━━━━━━━━ 74s 3ms/step - loss: 0.0017 - mae: 0.0237 - val_loss: 0.0014 - val_mae: 0.0209
Epoch 2/10
22876/22888 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0016 - mae: 0.0225
Epoch 2: val_loss improved from 0.00142 to 0.00137, saving model to /content/drive/MyDrive/T-MLP/saved_models/MLP_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/T-MLP/saved_model